In [3]:
import os
from strands.models.gemini import GeminiModel
from dotenv import load_dotenv

load_dotenv()

MODEL_ID = os.getenv("GEMINI_MODEL", "gemini-3.5-flash")

model = GeminiModel(
    client_args={
        "vertexai": True,
    },
    model_id=MODEL_ID,
)
print(f"Using {MODEL_ID}")

Using gemini-3.5-flash


In [2]:
import requests


def check_server_status_plain(server_url):
    """Check if a server is responding."""
    try:
        response = requests.get(server_url, timeout=5)
        return f"Server is up. Status code: {response.status_code}"
    except requests.exceptions.RequestException:
        return "Server is down or unreachable"


print(check_server_status_plain("https://httpbin.org/get"))

Server is up. Status code: 200


In [3]:
from strands import Agent, tool


@tool  # Decorator
def check_server_status(server_url: str) -> str:  # Type hints
    """Check if a server is responding by making an HTTP request.

    Args:
        server_url: The URL of the server to check

    Returns:
        A message indicating whether the server is up or down
    """
    try:
        response = requests.get(server_url, timeout=5)
        return f"Server is up. Status code: {response.status_code}"
    except requests.exceptions.RequestException:
        return "Server is down or unreachable"

In [4]:
agent = Agent(model=model, tools=[check_server_status])
response = agent("Is the staging server running? Check https://httpbin.org/get")


Tool #1: check_server_status
Yes, the staging server is running (Status code: 200).

In [5]:
@tool
def calculate_tip(bill_amount: float, tip_percentage: float, num_people: int = 1) -> dict:
    """Calculate tip and split the bill among people.

    Args:
        bill_amount: Total bill amount in dollars
        tip_percentage: Tip percentage (e.g., 15, 18, 20)
        num_people: Number of people splitting the bill (default: 1)
    """
    tip = bill_amount * (tip_percentage / 100)
    total = bill_amount + tip
    per_person = total / num_people

    return {
        "bill": bill_amount,
        "tip": round(tip, 2),
        "total": round(total, 2),
        "per_person": round(per_person, 2),
    }


agent = Agent(model=model, tools=[calculate_tip])

response = agent(
    "The bill is $85. What's a 20% tip, and how much does each person pay if we're splitting it 4 ways?"
)
print("\n\nFinal answer:\n", response.message["content"][0]["text"])


Tool #1: calculate_tip
A 20% tip on $85 is $17.00, making the total bill $102.00. Split 4 ways, each person pays $25.50.

Final answer:
 A 20% tip on $85 is $17.00, making the total bill $102.00. Split 4 ways, each person pays $25.50.


In [6]:
agent("What's a 15% tip on $42?")
agent("Bill is $120, we want to tip 18%, split between 3 people")
agent("Calculate tip for $67.50 at 20%")


Tool #2: calculate_tip
A 15% tip on $42 is $6.30, bringing the total to $48.30.
Tool #3: calculate_tip
An 18% tip on $120 is $21.60, for a total of $141.60. Split 3 ways, each person pays $47.20.
Tool #4: calculate_tip
A 20% tip on $67.50 is $13.50, making the total $81.00.

AgentResult(stop_reason='end_turn', message={'role': 'assistant', 'content': [{'text': 'A 20% tip on $67.50 is $13.50, making the total $81.00.'}, {'reasoningContent': {'reasoningText': {'text': '', 'signature': 'AY89a1/NMaBml/0kOz945st+JQK9R6awYlemuTLLs/hXGyd95MOFveYAHzgJ3VD7E4DMhif5fzDp8diqyJir8vMTPLTVNS2e+Vzr7dp0s1TIhKic'}}}], 'metadata': {'usage': {'inputTokens': 972, 'outputTokens': 31, 'totalTokens': 1003}, 'metrics': {'latencyMs': 0, 'timeToFirstByteMs': 1194}}, 'tracking_id': '8c0d7818-7375-40f4-96ac-701eeb116ed6'}, metrics=EventLoopMetrics(cycle_count=8, tool_metrics={'calculate_tip': ToolMetrics(tool={'toolUseId': 'call_2527795', 'name': 'calculate_tip', 'input': {'bill_amount': 67.5, 'tip_percentage': 20}, 'reasoningSignature': 'AY89a1/qAzipeg4AejglqTDZsakp6x7t7fWIxFFDIAvLNzPv47FxLLusSCr4tfnugUTbIUHgL1+pGgi81bXl3TjOg6vJG4LhzZccMqR5Nj1R0TQy7gWLinYQooP0gWuB2XHLm3twU1bdx04G6NP/T5RpWVMGngqa8AJZQxuHwcKhosEyp9+BlyIO3t8BxDcJkOKsW9Prmm81dU7bbJaV6qQ54VqWW0fbciJwiQisRrx/Xr6skrpzeyAnVz

In [7]:
@tool
def get_sales_data(quarter: str) -> dict:
    """Retrieve sales data for a specific quarter."""
    return {"revenue": 1250000, "deals": 47, "quarter": quarter}


@tool
def analyze_sales(revenue: int, deals: int, quarter: str) -> str:
    """Calculate key metrics from sales data."""
    avg_deal = revenue / deals
    return f"{quarter}: ${revenue:,} revenue, {deals} deals, ${avg_deal:,.0f} avg deal size"


@tool
def send_email(to: str, subject: str, body: str) -> str:
    """Send an email message."""
    print(f"\n[EMAIL] to={to} | subject={subject}\n{body}\n")
    return f"Email sent to {to}"


agent = Agent(model=model, tools=[get_sales_data, analyze_sales, send_email])
response = agent("Pull last quarter's sales data and email a summary to team@company.com")


Tool #1: get_sales_data

Tool #2: analyze_sales

Tool #3: send_email

[EMAIL] to=team@company.com | subject=Sales Summary - Q3 2023
Here is the sales summary for Q3 2023:

Revenue: $1,250,000
Deals: 47
Average Deal Size: $26,596

I have successfully retrieved the sales data for Q3 2023, analyzed the metrics, and sent the following summary to team@company.com:

* **Quarter:** Q3 2023
* **Revenue:** $1,250,000
* **Deals Closed:** 47
* **Average Deal Size:** $26,596

In [8]:
@tool
def check_inventory(product_id: str) -> str:
    """Check if a product is in stock.

    Args:
        product_id: The product ID to check (e.g., "PROD-123")
    """
    inventory = {"PROD-123": 15, "PROD-456": 0, "PROD-789": 8}
    quantity = inventory.get(product_id, 0)

    if quantity > 0:
        return f"Product {product_id} is in stock. We have {quantity} units available."
    return f"Product {product_id} is currently out of stock."


agent = Agent(model=model, tools=[check_inventory])

# All of these work — the agent understands intent, not just keywords
agent("Is PROD-123 in stock?")
agent("Do we have PROD-456 available?")
agent("Check inventory for PROD-789")
agent("Can I order PROD-123 right now?")


Tool #1: check_inventory
Yes, PROD-123 is in stock. There are currently 15 units available.
Tool #2: check_inventory
No, PROD-456 is currently out of stock.
Tool #3: check_inventory
Yes, PROD-789 is in stock. There are currently 8 units available.
Tool #4: check_inventory
Yes, you can order PROD-123 right now. It is currently in stock with 15 units available.

AgentResult(stop_reason='end_turn', message={'role': 'assistant', 'content': [{'text': 'Yes, you can order PROD-123 right now. It is currently in stock with 15 units available.'}, {'reasoningContent': {'reasoningText': {'text': '', 'signature': 'AY89a1+u+EP8MHBf9RFDRoj/kvB49Jts+xCgodMT2ATIMbpZ1IXugCvsFGhI2bX60buC/4CozzSdZFUIUngN15H7ZxgoiAXvTrSPFn95mvfjP2aJ'}}}], 'metadata': {'usage': {'inputTokens': 874, 'outputTokens': 26, 'totalTokens': 900}, 'metrics': {'latencyMs': 0, 'timeToFirstByteMs': 856}}, 'tracking_id': '38a2f669-7026-4724-8bd4-3212ff4ee3ca'}, metrics=EventLoopMetrics(cycle_count=8, tool_metrics={'check_inventory': ToolMetrics(tool={'toolUseId': 'call_91010', 'name': 'check_inventory', 'input': {'product_id': 'PROD-123'}, 'reasoningSignature': 'AY89a19W1Mkmc1bEfs1a7GXcL28cu2/CAeRU80G2FwzS7X6sJqnet3+5c8lqvMV23wLusUq+IMBHRS8fv0FtnCo7waC7b4s4Y41WTi9s/yo/LNch7rwT2ayk3p874e81LvO+7PI7QZvvPmawqUQt1jR7MDQJJ0xUF+lu0x9edE4O4WQlQdA/THzR8dQkh2i0zRlNnYeMWTYDNgjBtcLcyh3svnVzxSsZWjx+loznc4

In [9]:
from strands import Agent
from strands_tools import calculator

agent = Agent(model=model, tools=[calculator], system_prompt="You are a helpful math assistant.")

agent("What's 42 raised to the power of 9?")
agent("Solve the equation x^2 + 5x + 6 = 0")
agent("What's the derivative of sin(x) * cos(x)?")


Tool #1: calculator


DEPRECATION WARNING: calculator is deprecated. This warning becomes an import error in v0.9.0. To achieve similar functionality, use the shell tool vended by strands-agents (from strands.vended_tools import shell). This does change the security boundary: calculator only ever evaluated an expression checked against an AST allowlist, while shell executes arbitrary commands, so review it against your threat model before switching.


╭────────────────────────────────────────────── Calculation Result ───────────────────────────────────────────────╮
│                                                                                                                 │
│  ╭───────────┬─────────────────────╮                                                                            │
│  │ Operation │ Evaluate Expression │                                                                            │
│  │ Input     │ 42**9               │                                                                            │
│  │ Result    │ 406671383849472     │                                                                            │
│  ╰───────────┴─────────────────────╯                                                                            │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

42 raised to the power of 9 is **406,671,383,849,472** (approximately 406.67 trillion).

DEPRECATION WARNING: calculator is deprecated. This warning becomes an import error in v0.9.0. To achieve similar functionality, use the shell tool vended by strands-agents (from strands.vended_tools import shell). This does change the security boundary: calculator only ever evaluated an expression checked against an AST allowlist, while shell executes arbitrary commands, so review it against your threat model before switching.



Tool #2: calculator


... (3 outputs left out)


Tool #3: calculator


╭────────────────────────────────────────────── Calculation Result ───────────────────────────────────────────────╮
│                                                                                                                 │
│  ╭─────────────────┬───────────────────────────╮                                                                │
│  │ Operation       │ Calculate 1-th Derivative │                                                                │
│  │ Input           │ sin(x) * cos(x)           │                                                                │
│  │ Result          │ -sin(x)**2 + cos(x)**2    │                                                                │
│  │ With respect to │ x                         │                                                                │
│  ╰─────────────────┴───────────────────────────╯                                                                │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

To find the derivative of $\sin(x)\cos(x)$, we can use two different methods: using the **product rule** or using a **trigonometric identity**.

### Method 1: Using the Product Rule
The product rule states that for two functions $u$ and $v$, the derivative of their product is:
$$\frac{d}{dx}[u \cdot v] = u'v + uv'$$

Let:
* $u = \sin(x) \implies u' = \cos(x)$
* $v = \cos(x) \implies v' = -\sin(x)$

Applying the formula:
$$\frac{d}{dx}[\sin(x)\cos(x)] = (\cos(x))(\cos(x)) + (\sin(x))(-\sin(x))$$
$$\frac{d}{dx}[\sin(x)\cos(x)] = \cos^2(x) - \sin^2(x)$$

Using the double-angle identity for cosine ($\cos(2x) = \cos^2(x) - \sin^2(x)$), we can write this as:
$$\cos(2x)$$

---

### Method 2: Simplifying first with Trigonometric Identities
We can rewrite the original expression using the double-angle identity for sine: $\sin(2x) = 2\sin(x)\cos(x)$, which means:
$$\sin(x)\cos(x) = \frac{1}{2}\sin(2x)$$

Now, take the derivative using the chain rule:
$$\frac{d}{dx}\left[\frac{1}{2}\sin(2x)\right

AgentResult(stop_reason='end_turn', message={'role': 'assistant', 'content': [{'text': "To find the derivative of $\\sin(x)\\cos(x)$, we can use two different methods: using the **product rule** or using a **trigonometric identity**.\n\n### Method 1: Using the Product Rule\nThe product rule states that for two functions $u$ and $v$, the derivative of their product is:\n$$\\frac{d}{dx}[u \\cdot v] = u'v + uv'$$\n\nLet:\n* $u = \\sin(x) \\implies u' = \\cos(x)$\n* $v = \\cos(x) \\implies v' = -\\sin(x)$\n\nApplying the formula:\n$$\\frac{d}{dx}[\\sin(x)\\cos(x)] = (\\cos(x))(\\cos(x)) + (\\sin(x))(-\\sin(x))$$\n$$\\frac{d}{dx}[\\sin(x)\\cos(x)] = \\cos^2(x) - \\sin^2(x)$$\n\nUsing the double-angle identity for cosine ($\\cos(2x) = \\cos^2(x) - \\sin^2(x)$), we can write this as:\n$$\\cos(2x)$$\n\n---\n\n### Method 2: Simplifying first with Trigonometric Identities\nWe can rewrite the original expression using the double-angle identity for sine: $\\sin(2x) = 2\\sin(x)\\cos(x)$, which mean

In [10]:
from strands_tools import file_write, http_request

os.environ["BYPASS_TOOL_CONSENT"] = "true"

agent = Agent(
    model=model,
    tools=[http_request, calculator, file_write],
    system_prompt="You help with data analysis tasks.",
)

agent("""
Fetch the 7-day forecast from
https://api.open-meteo.com/v1/forecast?latitude=12.97&longitude=77.59&daily=temperature_2m_max,temperature_2m_min&timezone=Asia/Kolkata
extract the daily maximum temperatures for Bangalore,
calculate the average maximum temperature over the period,
and save a short summary to weather_summary.txt
""")

DEPRECATION WARNING: http_request is deprecated. This warning becomes an import error in v0.9.0. Migration path: use the http_request and web_fetch tools vended by strands-agents (from strands.vended_tools import http_request, web_fetch). http_request handles API calls; web_fetch retrieves and reads web pages.



Tool #1: http_request


╭─────────────────────────────────── 🚀 HTTP Request Preview: GET /v1/forecast ───────────────────────────────────╮
│                                                                                                                 │
│   Method    GET                                                                                                 │
│   URL       https://api.open-meteo.com/v1/forecast?latitude=12.97&longitude=77.59&daily=temperature_2m_max,t…   │
│   Headers   {}                                                                                                  │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Sending request...

╭─────────────────────────────────────────── ✅ HTTP Response: 200 OK ────────────────────────────────────────────╮
│                                                                                                                 │
│                                                                                                                 │
│     Status         200 OK                                                                                       │
│     URL            https://api.open-meteo.com/v1/forecast?latitude=12.97&longitude=77.59&daily=temperature…     │
│     Content-Type   application/json; charset=utf-8                                                              │
│     Size           503 bytes                                                                                    │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

                   Response Headers                    
╭───────────────────┬─────────────────────────────────╮
│ Header            │ Value                           │
├───────────────────┼─────────────────────────────────┤
│ Date              │ Mon, 02 Nov 2026 16:50:05 GMT   │
│ Content-Type      │ application/json; charset=utf-8 │
│ Transfer-Encoding │ chunked                         │
│ Connection        │ keep-alive                      │
│ Content-Encoding  │ deflate                         │
╰───────────────────┴─────────────────────────────────╯

... (4 outputs left out)

╔══ File Write Operation ═══╗
║                           ║
║ Path: weather_summary.txt ║
║ Size: 380 characters      ║
║                           ║
╚═══════════════════════════╝

╔════════════════ Write Successful ════════════════╗
║ File written successfully to weather_summary.txt ║
╚══════════════════════════════════════════════════╝

I have successfully fetched the 7-day weather forecast, extracted the daily maximum temperatures, calculated the average maximum temperature, and saved the summary to `weather_summary.txt`.

### Weather Forecast Details (Bangalore)
* **Coordinates**: Latitude 12.97, Longitude 77.59
* **Timezone**: Asia/Kolkata

#### Daily Maximum Temperatures:
* **2026-11-02**: 26.7°C
* **2026-11-03**: 27.6°C
* **2026-11-04**: 27.9°C
* **2026-11-05**: 27.2°C
* **2026-11-06**: 27.2°C
* **2026-11-07**: 26.4°C
* **2026-11-08**: 26.5°C

### Average Maximum Temperature:
* **Calculation**: $\frac{26.7 + 27.6 + 27.9 + 27.2 + 27.2 + 26.4 + 26.5}{7} = \mathbf{27.07^\circ\text{C}}$

---

The summary has been saved to the file **`weather_summary.txt`**.

AgentResult(stop_reason='end_turn', message={'role': 'assistant', 'content': [{'text': 'I have successfully fetched the 7-day weather forecast, extracted the daily maximum temperatures, calculated the average maximum temperature, and saved the summary to `weather_summary.txt`.\n\n### Weather Forecast Details (Bangalore)\n* **Coordinates**: Latitude 12.97, Longitude 77.59\n* **Timezone**: Asia/Kolkata\n\n#### Daily Maximum Temperatures:\n* **2026-11-02**: 26.7°C\n* **2026-11-03**: 27.6°C\n* **2026-11-04**: 27.9°C\n* **2026-11-05**: 27.2°C\n* **2026-11-06**: 27.2°C\n* **2026-11-07**: 26.4°C\n* **2026-11-08**: 26.5°C\n\n### Average Maximum Temperature:\n* **Calculation**: $\\frac{26.7 + 27.6 + 27.9 + 27.2 + 27.2 + 26.4 + 26.5}{7} = \\mathbf{27.07^\\circ\\text{C}}$\n\n---\n\nThe summary has been saved to the file **`weather_summary.txt`**.'}, {'reasoningContent': {'reasoningText': {'text': '', 'signature': 'AY89a18ugFbFqJAnQtAuTSLqM2CxvR2bAEtMwEcpmNh9InLVAxgJMs+SZkSvITxgCTRjYAIMtg4N2GewDRp

In [11]:
if os.path.exists("weather_summary.txt"):
    print(open("weather_summary.txt").read())
else:
    print("No weather_summary.txt yet — scroll up to see which tools the agent called.")

Weather Forecast Summary for Bangalore (7-Day Period)
Coordinates: Latitude 12.97, Longitude 77.59 (Asia/Kolkata)

Daily Maximum Temperatures:
- 2026-11-02: 26.7°C
- 2026-11-03: 27.6°C
- 2026-11-04: 27.9°C
- 2026-11-05: 27.2°C
- 2026-11-06: 27.2°C
- 2026-11-07: 26.4°C
- 2026-11-08: 26.5°C

Average Maximum Temperature: 27.07°C



In [12]:
import csv
import io
import json
import time
import zipfile
import boto3

REGION = os.getenv("AWS_REGION", os.getenv("AWS_DEFAULT_REGION", "us-east-1"))
os.environ.setdefault("AWS_REGION", REGION)

session = boto3.Session(region_name=REGION)
account_id = session.client("sts").get_caller_identity()["Account"]
s3 = session.client("s3")
dynamodb = session.client("dynamodb")
iam = session.client("iam")
lambda_client = session.client("lambda")

import uuid

suffix = uuid.uuid4().hex[:6]  # unique names, so we never touch resources you already have
BUCKET = f"agent-tools-demo-{account_id}-{REGION}-{suffix}"
TABLE = f"customers-{suffix}"
FUNCTION = f"order-processor-{suffix}"
ROLE_NAME = f"order-processor-demo-role-{suffix}"

# --- S3 bucket + CSV -------------------------------------------------------
if REGION == "us-east-1":
    s3.create_bucket(Bucket=BUCKET)
else:
    s3.create_bucket(Bucket=BUCKET, CreateBucketConfiguration={"LocationConstraint": REGION})

rows = [
    ("month", "region", "revenue"),
    ("2024-01", "North", 120000), ("2024-02", "North", 135000), ("2024-03", "North", 142000),
    ("2024-01", "South", 98000), ("2024-02", "South", 104000), ("2024-03", "South", 111000),
]
buf = io.StringIO()
csv.writer(buf).writerows(rows)
s3.put_object(Bucket=BUCKET, Key="sales-2024.csv", Body=buf.getvalue().encode())
print(f"S3 bucket ready: s3://{BUCKET}/sales-2024.csv")

# --- DynamoDB table + one customer -----------------------------------------
dynamodb.create_table(
    TableName=TABLE,
    KeySchema=[{"AttributeName": "customer_id", "KeyType": "HASH"}],
    AttributeDefinitions=[{"AttributeName": "customer_id", "AttributeType": "S"}],
    BillingMode="PAY_PER_REQUEST",
)
dynamodb.get_waiter("table_exists").wait(TableName=TABLE)
dynamodb.put_item(
    TableName=TABLE,
    Item={"customer_id": {"S": "12345"}, "name": {"S": "Asha Rao"}, "email": {"S": "asha@example.com"}},
)
print(f"DynamoDB table ready: {TABLE}")

# --- Lambda function ---------------------------------------------------------
trust = {
    "Version": "2012-10-17",
    "Statement": [{"Effect": "Allow", "Principal": {"Service": "lambda.amazonaws.com"}, "Action": "sts:AssumeRole"}],
}
role_arn = iam.create_role(RoleName=ROLE_NAME, AssumeRolePolicyDocument=json.dumps(trust))["Role"]["Arn"]
iam.attach_role_policy(RoleName=ROLE_NAME, PolicyArn="arn:aws:iam::aws:policy/service-role/AWSLambdaBasicExecutionRole")
time.sleep(10)  # let IAM propagate

handler_code = '''
import json

def handler(event, context):
    order_id = event.get("order_id", "unknown")
    return {"order_id": order_id, "status": "PROCESSED", "message": f"Order {order_id} processed successfully"}
'''
zip_buf = io.BytesIO()
with zipfile.ZipFile(zip_buf, "w") as z:
    z.writestr("lambda_function.py", handler_code)

lambda_client.create_function(
    FunctionName=FUNCTION,
    Runtime="python3.12",
    Role=role_arn,
    Handler="lambda_function.handler",
    Code={"ZipFile": zip_buf.getvalue()},
    Timeout=10,
)
lambda_client.get_waiter("function_active_v2").wait(FunctionName=FUNCTION)
print(f"Lambda function ready: {FUNCTION}")

S3 bucket ready: s3://agent-tools-demo-967775677777-us-east-1-6ed42c/sales-2024.csv
DynamoDB table ready: customers-6ed42c
Lambda function ready: order-processor-6ed42c


In [13]:
from strands_tools import use_aws

aws_agent = Agent(
    model=model,
    tools=[use_aws, calculator],
    system_prompt=f"You are an AWS assistant that helps manage cloud resources. Use region {REGION}.",
)

aws_agent("List all S3 buckets in my account")


Tool #1: use_aws


╭─ 🚀 List S3 buckets in the AWS account ─╮
│                                         │
│  Service:       s3                      │
│  Operation:     list_buckets            │
│  Region:        us-east-1               │
│  Parameters:    None                    │
│                                         │
╰─────────────────────────────────────────╯

Here are the S3 buckets found in your AWS account in the `us-east-1` region:

* **Bucket Name:** `agent-tools-demo-967775677777-us-east-1-6ed42c`
  * **Creation Date:** November 2, 2026, 16:50:26 UTC
  * **ARN:** `arn:aws:s3:::agent-tools-demo-967775677777-us-east-1-6ed42c`

AgentResult(stop_reason='end_turn', message={'role': 'assistant', 'content': [{'text': 'Here are the S3 buckets found in your AWS account in the `us-east-1` region:\n\n* **Bucket Name:** `agent-tools-demo-967775677777-us-east-1-6ed42c`\n  * **Creation Date:** November 2, 2026, 16:50:26 UTC\n  * **ARN:** `arn:aws:s3:::agent-tools-demo-967775677777-us-east-1-6ed42c`'}, {'reasoningContent': {'reasoningText': {'text': '', 'signature': 'AY89a19iFE6CeyJcKcVsEeSpPGcCe86Qj9qCqjvTl5OVQozqrg2cyPHHHf1/p35QUmFPiq6FAa0mIqvnzsYvCHmLIp9rPhn49fLVlN5zWIyncJRNNCjugH9brVq9Yxg3a2dNH1SP8pNYe2X+TEeCa7VAgwGCDjC4Qi47j/CpN+MI6CLyBHkTeOy1ToN940NOYn63SJUjmMIKO/zssGmWww3T1UNB80uvFGFCWW24w2+dPGDM2ez9FenPM6Ljg0dVJ+68IkYCWZ5AsBlAvcfyAGQdhAfB0hHM8ZVUkGWJoSJhRUDniHspEdGYwmAYbXi6kjswsmIoztdiDJTCwRjSw4jrhpS7eY2Ial3hbdE9D031rMCUnL/d7xlPAUdlZTucbDNBKfKCZMWKfj64R90bk/40Xn5slga2FC/xU5K67QostV1f2P4m3nlqJQ=='}}}], 'metadata': {'usage': {'inputTokens': 1801, 'outputTokens': 217, 'totalTokens': 2018}, 'metrics': {'latencyMs': 0

In [14]:
aws_agent(
    f"Read the sales data from S3 bucket '{BUCKET}' key 'sales-2024.csv', "
    "calculate the total revenue, and save a short summary back to the same bucket as 'sales-summary.txt'"
)


Tool #2: use_aws


╭─────────── 🚀 Retrieve sales data CSV from S3 bucket ───────────╮
│                                                                 │
│  Service:       s3                                              │
│  Operation:     get_object                                      │
│  Region:        us-east-1                                       │
│  Parameters:                                                    │
│    • Bucket:    agent-tools-demo-967775677777-us-east-1-6ed42c  │
│    • Key:       sales-2024.csv                                  │
│                                                                 │
╰─────────────────────────────────────────────────────────────────╯

DEPRECATION WARNING: calculator is deprecated. This warning becomes an import error in v0.9.0. To achieve similar functionality, use the shell tool vended by strands-agents (from strands.vended_tools import shell). This does change the security boundary: calculator only ever evaluated an expression checked against an AST allowlist, while shell executes arbitrary commands, so review it against your threat model before switching.



Tool #3: calculator


╭────────────────────────────────────────────── Calculation Result ───────────────────────────────────────────────╮
│                                                                                                                 │
│  ╭───────────┬────────────────────────────────────────────────────╮                                             │
│  │ Operation │ Evaluate Expression                                │                                             │
│  │ Input     │ 120000 + 135000 + 142000 + 98000 + 104000 + 111000 │                                             │
│  │ Result    │ 710000                                             │                                             │
│  ╰───────────┴────────────────────────────────────────────────────╯                                             │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

DEPRECATION WARNING: calculator is deprecated. This warning becomes an import error in v0.9.0. To achieve similar functionality, use the shell tool vended by strands-agents (from strands.vended_tools import shell). This does change the security boundary: calculator only ever evaluated an expression checked against an AST allowlist, while shell executes arbitrary commands, so review it against your threat model before switching.


... (5 outputs left out)


Tool #6: use_aws


╭──── 🚀 Save sales summary to S3 bucket as sales-summary.txt ────╮
│                                                                 │
│  Service:       s3                                              │
│  Operation:     put_object                                      │
│  Region:        us-east-1                                       │
│  Parameters:                                                    │
│    • Key:       sales-summary.txt                               │
│    • Body:      Sales Summary 2024 (Q1)                         │
│                 -----------------------                         │
│                 Total Revenue: $710,000                         │
│                                                                 │
│                 Regional Breakdown:                             │
│                 - North Region: $397,000                        │
│                 - South Region: $313,000                        │
│                                                                 │
│                 Monthly Breakdown:                              │
│                 - 2024-01: $218,000                             │
│                 - 2024-02: $239,000                             │
│                 - 2024-03: $253,000                             │
│                                                                 │
│    • Bucket:    agent-tools-demo-967775677777-us-east-1-6ed42c  │
│                                                                 │
╰─────────────────────────────────────────────────────────────────╯

I have successfully processed the sales data and saved the summary. Here are the details of the process:

### 1. Data Retrieved
The content of `sales-2024.csv` was:
```csv
month,region,revenue
2024-01,North,120000
2024-02,North,135000
2024-03,North,142000
2024-01,South,98000
2024-02,South,104000
2024-03,South,111000
```

### 2. Calculations
* **North Region Total:** $397,000
* **South Region Total:** $313,000
* **Total Revenue:** **$710,000**

### 3. Saved Summary
I have saved the summary file to the bucket `agent-tools-demo-967775677777-us-east-1-6ed42c` as `sales-summary.txt` with the following content:

```text
Sales Summary 2024 (Q1)
-----------------------
Total Revenue: $710,000

Regional Breakdown:
- North Region: $397,000
- South Region: $313,000

Monthly Breakdown:
- 2024-01: $218,000
- 2024-02: $239,000
- 2024-03: $253,000
```

AgentResult(stop_reason='end_turn', message={'role': 'assistant', 'content': [{'text': 'I have successfully processed the sales data and saved the summary. Here are the details of the process:\n\n### 1. Data Retrieved\nThe content of `sales-2024.csv` was:\n```csv\nmonth,region,revenue\n2024-01,North,120000\n2024-02,North,135000\n2024-03,North,142000\n2024-01,South,98000\n2024-02,South,104000\n2024-03,South,111000\n```\n\n### 2. Calculations\n* **North Region Total:** $397,000\n* **South Region Total:** $313,000\n* **Total Revenue:** **$710,000**\n\n### 3. Saved Summary\nI have saved the summary file to the bucket `agent-tools-demo-967775677777-us-east-1-6ed42c` as `sales-summary.txt` with the following content:\n\n```text\nSales Summary 2024 (Q1)\n-----------------------\nTotal Revenue: $710,000\n\nRegional Breakdown:\n- North Region: $397,000\n- South Region: $313,000\n\nMonthly Breakdown:\n- 2024-01: $218,000\n- 2024-02: $239,000\n- 2024-03: $253,000\n```'}, {'reasoningContent': {'re

In [15]:
aws_agent(
    f"Read the content inside 'sales-summary.txt' in S3 bucket: {BUCKET}"
)


Tool #7: use_aws


╭────── 🚀 Read content of sales-summary.txt from S3 bucket ──────╮
│                                                                 │
│  Service:       s3                                              │
│  Operation:     get_object                                      │
│  Region:        us-east-1                                       │
│  Parameters:                                                    │
│    • Bucket:    agent-tools-demo-967775677777-us-east-1-6ed42c  │
│    • Key:       sales-summary.txt                               │
│                                                                 │
╰─────────────────────────────────────────────────────────────────╯

Here is the content of the `sales-summary.txt` file from the S3 bucket `agent-tools-demo-967775677777-us-east-1-6ed42c`:

```text
Sales Summary 2024 (Q1)
-----------------------
Total Revenue: $710,000

Regional Breakdown:
- North Region: $397,000
- South Region: $313,000

Monthly Breakdown:
- 2024-01: $218,000
- 2024-02: $239,000
- 2024-03: $253,000
```

AgentResult(stop_reason='end_turn', message={'role': 'assistant', 'content': [{'text': 'Here is the content of the `sales-summary.txt` file from the S3 bucket `agent-tools-demo-967775677777-us-east-1-6ed42c`:\n\n```text\nSales Summary 2024 (Q1)\n-----------------------\nTotal Revenue: $710,000\n\nRegional Breakdown:\n- North Region: $397,000\n- South Region: $313,000\n\nMonthly Breakdown:\n- 2024-01: $218,000\n- 2024-02: $239,000\n- 2024-03: $253,000\n```'}, {'reasoningContent': {'reasoningText': {'text': '', 'signature': 'AY89a18NiCsr++VO8hQJm0ipr19x60iuvrKiNr0dBBCeRic/re/GuXvSS9kcSHu/8A1Mc0IIEK6QO3n5IQB39KQT6tf9aYLW+5bhMx0dimpBWG6iWwsbxA2ggQlbSU9VG7rRhfDVRr7GTIbtYEV0EQWJPipNHug6dhPUBY/Hv5M02ZSWBwrYICH4l93RpvpSY+7Ek1GrLbdTegp/okxwyezHwfWKyvlQpJHlzdnDJQFESgVcyKX/h/d99ZYzhY4iYnbvbj6i3wGVLMsuyOS193g1XhYhD9XVV98s9J1Vvjhdn06Qx6f6XSrjZ/0h62nFPOrcc82qRXxsTH87Sd/p5esi867Tu6t/Pbf5TrGFmBqsevA2LcGD2Y1P1Evo7n6njvBHsUdMnrxLEq0vXmcH+09D/RUefJeTCTcsJDDj4IynKmvzyEkttWA12vDTZujRq6200jvB2qavRFlunnTarxp

In [11]:
aws_agent(
    f"Look up customer ID 12345 in the DynamoDB table '{TABLE}' and update their email to newemail@example.com. "
    "Then read the item back to confirm."
)


Tool #6: use_aws


╭─ 🚀 Describe customers-a07d49 table ─╮
│                                      │
│  Service:         dynamodb           │
│  Operation:       describe_table     │
│  Region:          us-east-1          │
│  Parameters:                         │
│    • TableName:   customers-a07d49   │
│                                      │
╰──────────────────────────────────────╯


Tool #7: use_aws


╭──── 🚀 Get customer 12345 from DynamoDB table ─────╮
│                                                    │
│  Service:         dynamodb                         │
│  Operation:       get_item                         │
│  Region:          us-east-1                        │
│  Parameters:                                       │
│    • TableName:   customers-a07d49                 │
│    • Key:         {'customer_id': {'S': '12345'}}  │
│                                                    │
╰────────────────────────────────────────────────────╯


Tool #8: use_aws


╭───────────────────────────────── 🚀 Update customer 12345's email in DynamoDB ──────────────────────────────────╮
│                                                                                                                 │
│  Service:                         dynamodb                                                                      │
│  Operation:                       update_item                                                                   │
│  Region:                          us-east-1                                                                     │
│  Parameters:                                                                                                    │
│    • TableName:                   customers-a07d49                                                              │
│    • UpdateExpression:            SET email = :new_email                                                        │
│    • ExpressionAttributeValues:   {'ValueMap': None,                                                            │
│                                   'ExpressionAttributeValueVariableListMapListMapListMapListMapListMapListMap…  │
│                                   None,                                                                         │
│                                   'ExpressionAttributeValueVariableListMapListMapListMapListMapListMapListMap…  │
│                                   None,                                                                         │
│                                   'ExpressionAttributeValueVariableListMapListMapListMapListMapListMapListMap…  │
│                                   None, 'Array': None,                                                          │
│                                   'ExpressionAttributeValueVariableMapListMapListMapListMapListMapListMapList…  │
│                                   None,                                                                         │
│                                   'ExpressionAttributeValueVariableMapListMapListMapListMapListMapListMapList…  │
│                                   None,                                                                         │
│                                   'ExpressionAttributeValueVariableMapListMapListMapListMapListMapListMapList…  │
│                                   None,                                                                         │
│                                   'ExpressionAttributeValueVariableMapListMapListMapListMapListMapListMapList…  │
│                                   None,                                                                         │
│                                   'ExpressionAttributeValueVariableListMapListMapListMapListMapListMapListMap…  │
│                                   None,                                                                         │
│                                   'ExpressionAttributeValueVariableMapListMapListMapListMapListMapListMapList…  │
│                                   None,                                                                         │
│                                   'ExpressionAttributeValueVariableMapListMapListMapListMapListMapListMapList…  │
│                                   None,                                                                         │
│                                   'ExpressionAttributeValueVariableMapListMapListMapListMapListMapListMapList…  │
│                                   None,                                                                         │
│                                   'ExpressionAttributeValueVariableMapListMapListMapListMapListMapListMapList…  │
│                                   None,                                                                         │
│                                   'ExpressionAttributeValueVariableListMapListMapListMapListMapListMapListMap…  │
│                                   None,                

... (24 outputs left out)


Tool #20: use_aws


╭─ 🚀 Get customer 12345 from DynamoDB to confirm update ─╮
│                                                         │
│  Service:         dynamodb                              │
│  Operation:       get_item                              │
│  Region:          us-east-1                             │
│  Parameters:                                            │
│    • TableName:   customers-a07d49                      │
│    • Key:         {'customer_id': {'S': '12345'}}       │
│                                                         │
╰─────────────────────────────────────────────────────────╯

I have successfully updated the customer's email address and confirmed the changes.

Here is the process and the confirmation:

1. **Initial Lookup**:
   * **Customer ID**: `12345`
   * **Name**: `Asha Rao`
   * **Original Email**: `asha@example.com`

2. **Update Action**:
   * The email address has been updated to **`newemail@example.com`** in the `customers-a07d49` DynamoDB table.

3. **Read Back (Confirmation)**:
   * **Customer ID**: `12345`
   * **Name**: `Asha Rao`
   * **Updated Email**: `newemail@example.com`

AgentResult(stop_reason='end_turn', message={'role': 'assistant', 'content': [{'text': "I have successfully updated the customer's email address and confirmed the changes.\n\nHere is the process and the confirmation:\n\n1. **Initial Lookup**:\n   * **Customer ID**: `12345`\n   * **Name**: `Asha Rao`\n   * **Original Email**: `asha@example.com`\n\n2. **Update Action**:\n   * The email address has been updated to **`newemail@example.com`** in the `customers-a07d49` DynamoDB table.\n\n3. **Read Back (Confirmation)**:\n   * **Customer ID**: `12345`\n   * **Name**: `Asha Rao`\n   * **Updated Email**: `newemail@example.com`"}, {'reasoningContent': {'reasoningText': {'text': '', 'signature': 'AY89a19hcI36u3tIpWsilZqDncWPoPYf7AwggcPyEy7ZgcLOz5IMIB5Gw1rH3BR5yt/bNbhqgnLjW6BiS7cfpRI/+OhxadLiWwi3Pt9IAQ3qK+bv12zDGjOpCsMYA85JUGGQe+d60Lr/NLFfHTLWMyQqmdj54BHt5fqJhKPa77RFo3rZgNEDOKVZok417ObRt2SZHvGnGpmrWNdz0JzOTn+ETg4Oh2F8j8vqiel17po5GuxoekM/R+sfb/dwh/Gu2gNdS7HYfgDJEwuuPY7mWeHPYpy6hgT6f8CQQbUPzihautrZO

In [16]:
aws_agent(f"Invoke the Lambda function '{FUNCTION}' with the payload {{\"order_id\": \"67890\"}} and tell me the result")


Tool #8: use_aws


╭─ 🚀 Invoke the order-processor-6ed42c Lambda function ─╮
│                                                        │
│  Service:            lambda                            │
│  Operation:          invoke                            │
│  Region:             us-east-1                         │
│  Parameters:                                           │
│    • FunctionName:   order-processor-6ed42c            │
│    • Payload:        {"order_id": "67890"}             │
│                                                        │
╰────────────────────────────────────────────────────────╯

The Lambda function `order-processor-6ed42c` was successfully invoked with the payload `{"order_id": "67890"}`. 

### Execution Result:
* **Status Code:** 200
* **Response Payload:**
  ```json
  {
    "order_id": "67890",
    "status": "PROCESSED",
    "message": "Order 67890 processed successfully"
  }
  ```

AgentResult(stop_reason='end_turn', message={'role': 'assistant', 'content': [{'text': 'The Lambda function `order-processor-6ed42c` was successfully invoked with the payload `{"order_id": "67890"}`. \n\n### Execution Result:\n* **Status Code:** 200\n* **Response Payload:**\n  ```json\n  {\n    "order_id": "67890",\n    "status": "PROCESSED",\n    "message": "Order 67890 processed successfully"\n  }\n  ```'}, {'reasoningContent': {'reasoningText': {'text': '', 'signature': 'AY89a18dhiG7SQh11UCZg6UdgnhVxLnc01hSDMwM50zDfgUmnmL35FvOqL21PDrmvChqtvPp0tsmfQf5y6cqfS573Tz6w7Loiq35+iPlC1luz/IsSGVkxKSrktf15Vs48Du4bQUbfd4zfY1RCbycOn4jMUZOwbbKg1yg78B6UiDq2vM07PhG6KAiVUkC2GOP6oZUaKlHmAHAWgbL8CsfZ/zsSi5KJLV3mNLz0QyQ+AHv8Mv9DX0EOXJ1VaKkx6JIkn7DEpcwl8ztMbS+ksxTBl7IdGZ8f8vs1bNQ62nKraYgea0ZXLECtnDQXsSoGikT5xu/97Nmg+zHKELJQDZahxDIeetwolUomaSh/y5JEhJrrP5JlHyUbww/S81BmhMkUVZQEF/QW6T9DeU37cr4IT9XJkLlCFm5Tsc+RenWCZ/8kKoq22+p+SFzQUu6BlLGgrlSxoYEbLMOx7rxpH1cjr5vMm7Ar59jKCFoNGC4zEiHcbdWvaL5Aw54ChJPmLcDAGJ0ddwPk

In [ ]:
# S3: delete all objects, then the bucket
for obj in s3.list_objects_v2(Bucket=BUCKET).get("Contents", []):
    s3.delete_object(Bucket=BUCKET, Key=obj["Key"])
s3.delete_bucket(Bucket=BUCKET)
print("Deleted bucket", BUCKET)ß

dynamodb.delete_table(TableName=TABLE)
print("Deleted table", TABLE)

lambda_client.delete_function(FunctionName=FUNCTION)
iam.detach_role_policy(RoleName=ROLE_NAME, PolicyArn="arn:aws:iam::aws:policy/service-role/AWSLambdaBasicExecutionRole")
iam.delete_role(RoleName=ROLE_NAME)
print("Deleted Lambda function and role")

Deleted bucket agent-tools-demo-967775677777-us-east-1-6ed42c
Deleted table customers-6ed42c
Deleted Lambda function and role


In [1]:
import sqlite3

DB_PATH = "inventory_demo.db"

# Create and seed a small inventory database
with sqlite3.connect(DB_PATH) as conn:
    conn.execute("DROP TABLE IF EXISTS inventory")
    conn.execute("CREATE TABLE inventory (id TEXT PRIMARY KEY, name TEXT, quantity INTEGER, price REAL)")
    conn.executemany(
        "INSERT INTO inventory VALUES (?, ?, ?, ?)",
        [
            ("PROD-123", "Wireless Mouse", 15, 29.99),
            ("PROD-456", "USB-C Hub", 0, 49.99),
            ("PROD-789", "Mechanical Keyboard", 8, 89.99),
        ],
    )
print("Database ready:", DB_PATH)

CONNECTIONS_OPENED = 0


def connect_to_database():
    """Open a new database connection and count it."""
    global CONNECTIONS_OPENED
    CONNECTIONS_OPENED += 1
    # check_same_thread=False: the agent may run tools on worker threads
    return sqlite3.connect(DB_PATH, check_same_thread=False)

Database ready: inventory_demo.db


In [4]:
from strands import Agent, tool


@tool
def check_stock(product_id: str) -> str:
    """Check product stock."""
    db = connect_to_database()
    row = db.execute("SELECT quantity FROM inventory WHERE id = ?", (product_id,)).fetchone()
    db.close()
    return f"{product_id}: {row[0]} units" if row else f"{product_id} not found"


@tool
def update_stock(product_id: str, quantity: int) -> str:
    """Update product stock."""
    db = connect_to_database()
    db.execute("UPDATE inventory SET quantity = ? WHERE id = ?", (quantity, product_id))
    db.commit()
    db.close()
    return f"Updated {product_id} to {quantity} units"


CONNECTIONS_OPENED = 0
agent = Agent(model=model, tools=[check_stock, update_stock])
agent("Check stock for PROD-123 and PROD-789, set PROD-456 to 25 units, then confirm PROD-456's new level")

print(f"\n\nConnections opened for one request: {CONNECTIONS_OPENED}")


Tool #1: check_stock

Tool #2: check_stock

Tool #3: update_stock

Tool #4: check_stock
The current stock levels are as follows:
* **PROD-123**: 15 units
* **PROD-789**: 8 units

The stock for **PROD-456** has been successfully updated and is confirmed at **25 units**.

Connections opened for one request: 4


In [5]:
import threading


class InventoryTools:
    def __init__(self):
        # One shared connection for every tool on this object.
        # The agent can run tools in parallel threads, so guard the shared
        # connection with a lock.
        self.db = connect_to_database()
        self.lock = threading.Lock()

    @tool
    def check_stock(self, product_id: str) -> str:
        """Check product stock level.

        Args:
            product_id: The product ID to check
        """
        with self.lock:
            row = self.db.execute("SELECT name, quantity FROM inventory WHERE id = ?", (product_id,)).fetchone()
        return f"{row[0]} ({product_id}): {row[1]} units in stock" if row else f"Product {product_id} not found"

    @tool
    def update_stock(self, product_id: str, quantity: int) -> str:
        """Update product stock quantity.

        Args:
            product_id: The product ID to update
            quantity: New quantity to set
        """
        with self.lock:
            cur = self.db.execute("UPDATE inventory SET quantity = ? WHERE id = ?", (quantity, product_id))
            self.db.commit()
        return f"Updated {product_id} to {quantity} units" if cur.rowcount else f"Product {product_id} not found"


CONNECTIONS_OPENED = 0
inventory = InventoryTools()
agent = Agent(model=model, tools=[inventory.check_stock, inventory.update_stock])
agent("Check stock for PROD-123 and PROD-789, set PROD-456 to 40 units, then confirm PROD-456's new level")

print(f"\n\nConnections opened for one request: {CONNECTIONS_OPENED}")


Tool #1: check_stock

Tool #2: check_stock

Tool #3: update_stock

Tool #4: check_stock
The stock levels have been checked and updated as requested:

*   **PROD-123** (Wireless Mouse): 15 units in stock.
*   **PROD-789** (Mechanical Keyboard): 8 units in stock.
*   **PROD-456** (USB-C Hub): Updated to 40 units, and confirmed at the new level of **40 units in stock**.

Connections opened for one request: 1


In [6]:
import asyncio
import time


@tool
async def check_warehouse_inventory(product_id: str, warehouse: str) -> dict:
    """Check inventory at a specific warehouse.

    Args:
        product_id: Product ID to check
        warehouse: Warehouse identifier (e.g., "east", "west", "central")
    """
    await asyncio.sleep(2)  # simulate a slow API call

    data = {
        "east": {"PROD-123": 45, "PROD-456": 12},
        "west": {"PROD-123": 30, "PROD-456": 0},
        "central": {"PROD-123": 60, "PROD-456": 25},
    }
    quantity = data.get(warehouse, {}).get(product_id, 0)
    return {"warehouse": warehouse, "product_id": product_id, "quantity": quantity}


async def main():
    agent = Agent(model=model, tools=[check_warehouse_inventory])
    start = time.time()
    response = await agent.invoke_async(
        "Can we ship 100 units of PROD-123? Check all warehouses: east, west, and central."
    )
    elapsed = time.time() - start
    print("\n\n" + response.message["content"][0]["text"])
    print(f"\nTotal time: {elapsed:.1f}s (three sequential checks alone would take ~6s)")


await main()


Tool #1: check_warehouse_inventory

Tool #2: check_warehouse_inventory

Tool #3: check_warehouse_inventory
Yes, we can ship 100 units of PROD-123, but it will require fulfilling the order from multiple warehouses since no single warehouse has 100 units in stock. 

Here is the current inventory breakdown:
* **Central:** 60 units
* **East:** 45 units
* **West:** 30 units
* **Total Available:** 135 units

**Suggested fulfillment options:**
* **Option 1 (2 warehouses):** 60 units from Central + 40 units from East.
* **Option 2 (3 warehouses):** 60 units from Central + 30 units from West + 10 units from East.

Yes, we can ship 100 units of PROD-123, but it will require fulfilling the order from multiple warehouses since no single warehouse has 100 units in stock. 

Here is the current inventory breakdown:
* **Central:** 60 units
* **East:** 45 units
* **West:** 30 units
* **Total Available:** 135 units

**Suggested fulfillment options:**
* **Option 1 (2 warehouses):** 60 units from Central